---

# 📘 두 설계 코드(`Step 1+2` vs `v2_clear`)의 단계별 핵심 기법 비교 해설서

이 문서는 두 가지 서로 다른 접근 방식을 가진 코드들이 README의 각 단계에서 어떤 차별화된 알고리즘을 사용했는지 상세히 비교 분석합니다.

---

## [공통 구간] Step 1 ~ Step 2: 데이터 기반 다지기

두 파일 모두 초기 단계에서는 데이터의 물리적 타당성을 확보하고 가상 데이터를 생성하는 데 동일한 기법을 공유합니다.

### 1. 절댓값 Max Peak 추출 (`abs().idxmax()`)

* **기법 소개:** 시계열 데이터에서 압축(-)과 인장(+)의 방향에 상관없이 가장 파괴적인 에너지가 발생한 지점을 특정하는 전처리 방식입니다.
* **주요 특징:** 단순 최댓값만 찾을 경우 놓칠 수 있는 '치명적 압축 응력'을 보존하기 위해 절댓값을 기준으로 위치를 찾고, 원본의 부호를 다시 가져옵니다.
* **프로젝트 도입 이유:** 열 변형 과정에서 냉각 시 발생하는 압축 응력이 제품 파괴의 주원인이 될 수 있으므로, 이를 누락 없이 학습 데이터에 반영하기 위함입니다.

### 2. XGBoost 기반 대리 모델 (Surrogate Model)

* **기법 소개:** 결정 트리(Decision Tree) 기반의 강력한 앙상블 학습 알고리즘으로, 회귀 예측 속도가 매우 빠릅니다.
* **주요 특징:** ~900개의 시뮬레이션 데이터를 학습하여 수십만 개의 가상 설계 조합 결과를 1초 내에 산출합니다.
* **프로젝트 도입 이유:** CAE 시뮬레이션의 시간적 한계를 극복하고, 역설계 및 최적화에 필요한 방대한 '가상 데이터 풀'을 구축하기 위해 도입했습니다.

---

## [분기 구간 A] `Step 1 XGBoost + Step 2 (4).ipynb`: 설명 가능한 역설계

이 코드는 README의 **[Step 4]**에 집중하여, 타겟 곡선으로부터 설계를 추론하고 그 이유를 설명하는 데 특화되어 있습니다.

### 3. 1D-CNN 역방향 모델 (Inverse Mapping)

* **기법 소개:** 타겟 응력/변형 곡선(1차원 시계열 데이터)을 입력받아, 이를 구현할 수 있는 물리적 수치($P1 \sim P6$)를 출력하는 딥러닝 모델입니다.
* **작동 원리:** 다채널 텐서 형태로 입력된 파형의 특징을 CNN 렌즈로 훑어내어 최적의 설계 변수 조합을 단번에 예측합니다.
* **프로젝트 도입 이유:** 엔지니어가 원하는 '목표 성능'을 제시했을 때, AI가 즉각적으로 설계 초안(Draft)을 제안하도록 만들기 위함입니다.

### 4. 구조적 진화 분석 (Structural Evolution Analysis)

* **기법 소개:** AI가 도출한 초안의 구조적 변화를 시각화하고 물리적 통찰과 비교하는 분석 기법입니다.
* **주요 특징:** 초안과 최종안의 단면 구조를 대조하여 변수 간의 증감 관계를 설명합니다.
* **프로젝트 도입 이유:** "위쪽($P1$)은 얇게, 하단($P4, P6$)은 보강하는 레시피"와 같이 AI의 설계 의도를 인간 엔지니어가 물리적으로 이해(XAI)할 수 있도록 돕기 위해 도입했습니다.

---

## [분기 구간 B] `Flipchip surrogate v2_clear.ipynb`: 엔지니어링 최적화

이 코드는 README의 **[Step 5]**로 확장되어, 수학적으로 가장 완벽한 랭킹 1위 설계안을 확정하는 데 특화되어 있습니다.

### 5. NSGA-II (다목적 유전 알고리즘)

* **기법 소개:** 적자생존의 원리를 이용해 상충하는 두 목적(뒤틀림 최소화 vs 박리 최소화)을 동시에 만족하는 해를 찾는 알고리즘입니다.
* **작동 원리:** 수만 개의 개체를 세대별로 진화시키며 파레토 프론티어(Pareto Frontier)를 형성하여 최적의 집단을 선별합니다.
* **프로젝트 도입 이유:** 단순한 초안 출력을 넘어, 물리적 제약 조건을 한 치의 오차 없이 만족하는 '최종 확정안'을 계산하기 위해 사용했습니다.

### 6. GPR(가우시안 과정 회귀) 기반 강건 최적화

* **기법 소개:** 예측값과 함께 '불확실성($\sigma$)'을 동시에 제공하는 베이지안 머신러닝 기법입니다.
* **작동 원리:** '$\mu + 2\sigma < Limit$' 수식을 적용하여, AI가 예측에 자신이 없는 미지의 영역에서도 보수적으로 안전한 설계를 채택하게 합니다.
* **프로젝트 도입 이유:** 실제 공정 오차나 AI의 오차를 고려하더라도 제품이 파손되지 않는 '강건한(Robust)' 설계 수치를 도출하기 위해 도입했습니다.

### 7. Knee Point (최종 밸런스 점 추출)

* **기법 소개:** 파레토 곡선 상에서 두 목표 지표가 가장 황금 밸런스를 이루는 지점을 수학적으로 특정하는 기법입니다.
* **작동 원리:** 정규화된 목적 함수 공간에서 이상향(Origin)과의 거리가 가장 짧은 점을 찾아 Rank 1로 지정합니다.
* **프로젝트 도입 이유:** 수많은 최적 후보 중 의사결정권자에게 추천할 수 있는 단 하나의 '가장 합리적인 설계안'을 자동으로 결정하기 위함입니다.

---

### 💡 요약: 두 코드의 지향점 차이

* **`Step 1+2` 파일:** AI의 설계 논리를 시각적으로 증명하고 **해설**하는 리포트 도구.
* **`v2_clear` 파일:** 실제 생산에 투입 가능한 정밀한 **수치 확정** 및 최적화 시스템.